# Machine Learning Vegetation Indices

Vegetation indices are combinations of spectral bands (Blue, Green, Red, Infrared, etc) used in remote sensing to estimate properties of plant life, especially chlorophyll content. They are used to monitor plant health, climate trends, biomass, etc. In machine learning terms, they would be considered *feature representations*, i.e. transformations of raw inputs into more informative representations of those inputs.

One of the most exciting promises of machine learning, and in particular deep learning, is the ability of models to build their own feature representations from data. This is what allows machine learning models to learn very powerful and *context-specific* functions to solve different tasks.

An interesting question therefore is whether machine learning models trained on remote sensing tasks for plant life would learn the same feature representations as human-derived vegetation indices or whether they would differ. And if they differ, does this represent a weakness in the machine learning model, e.g. a spurious data correlation that could lead to poor generalization, or an alternative but viable machine-learned vegetation index (MLVI) that could improve remote sensing performance and/or give new insight into the relationship between frequency response and plant life?

## Purpose of Study and Related Work

Some studies have used machine learning models to optimize or create new vegetation indices, e.g. DeepIndices by Albarracín et al. (2020) and Genetic-Programming-based Vegetation Indices (GPVI) by Vayssade et al. (2021). Although the current research could be used to a similar end, its focus is different. Rather than try to learn an "optimal" vegetation index, it asks the following questions: given the same information as a vegetation index (i.e. the same spectral bands and spatial area) and a remote sensing vegetation task (e.g. plant segmentation), does a machine learning model recreate the relevant vegetation index or learn a new feature representation of its own? If the latter, how can we understand this new representation, its causes and consequences for the task at hand? As such, it is more related to AI interpretation than optimization, though by the nature of machine learning its results could also be used for the latter.

# Pilot Study: Machine Learning NDVI

To give a proof of concept, I conduct a pilot study on machine learning the Normalized Difference Vegetation Index (NDVI), one of the most widely-used vegetation indices. I give a description of the methods and results below, with more details provided in the Jupyter notebook, where readers can reproduce the experiments and adjust their parameters.

NDVI is calculated as the difference between the Near-Infrared (NIR) and Red bands over their sum, as written out below,

$NDVI = \frac{NIR-Red}{NIR+Red}$.

Values range from $-1$ to $+1$, with higher values typically corresponding to healthier vegetation and lower values to stressed or absent vegetation.

I train a feedforward neural network (FNN) that takes two inputs, a Red value and an NIR value, and calculates a single, task-specific output. It has three hidden layers with 30 neurons per layer, for a total of 91 neurons including output (2911 trainable parameters). I train it on two tasks, described below.

## Task 1: Learning NDVI via Regression

For the first task, I fed the machine learning model Red and NIR values to see if it could directly predict NDVI values from them. Note that this is not the task proposed in the "Purpose of Study," where the model is supposed to learn the vegetation index indirectly through doing a remote sensing task, but I wanted to see first if it was even possible for the FNN to approximate NDVI. Given that the equation for NDVI is a rational function of Red and NIR, and an FNN can only represent piecewise-affine functions of its inputs (Balestriero & Baraniuk, 2018a,b), it might seem difficult to learn a close approximation. However, as Telgarsky (2017) finds, rational functions and neural networks are actually quite efficient at approximating each other! This means you don't need "too many" neurons in a neural network to approximate a rational function and vice-versa (specifically, each can approximate the other to $\epsilon$-precision with $O(poly \; log(1/\epsilon))$ degree or size). And indeed, at least to visual inspection, the FNN reproduces the NDVI function over all possible values of Red and NIR reasonably well.

## Task 2: Learning NDVI Thresholds via Classification

For the second task, I trained the machine learning model on actual images of plants with dirt / soil backgrounds. More specifically, I fed it specific pixels with only Red and NIR values and asked it to classify the pixel as plant or background. Carried out over all pixels in an image, this task is called "semantic segmentation" in machine learning literature.

The purpose was to see if the FNN would learn something equivalent to an NDVI threshold to classify pixels as either plant or background. Given NDVI's higher values corresponding to healthy plant life and lower values for barren ground, a naive approach to this task would be to simply set a threshold in NDVI and classify any pixels above that threshold as plant life and any pixels below it as background. (I note that I haven't actually seen this approach used in the literature, and I'm sure there are more effective methods to carry out this task; however, as the following experiments show, it isn't an unreasonable baseline!) If the FNN reproduced such an NDVI threshold, it would lend evidence to the efficacy of this method, and if it differed from any such threshold, it would be interesting to see where it differed and why.

It turns out that, while calculating NDVI is a rational function of the Red and NIR bands, calculating all possible Red and NIR values for a fixed threshold of NDVI is a linear function!$^{a}$ Specifically, for a fixed NDVI threshold $t$, you can calculate

$NIR = \frac{1+t}{1-t}Red \; \; \;$ or $\; \; \; Red = \frac{1-t}{1+t}NIR$.

($^{a}$This is obvious from the structure of the rational equation, I just hadn't thought of it before graphing out the function.)

A linear function should be easy for a neural network to learn, so there's no question of the FNN having the capacity to learn this function; it would come down to whether this was the most effective way of segmenting the data. To train the model, I selected 80,000 pixels from 80 images (40,000 vegetation, 40,000 background), and I set aside 11,000 pixels from a separate set of 11 images (5,500 vegetation, 5,500 background) for evaluation. To find the most effective NDVI threshold for classification, I did a grid search from $-1$ to $+1$ for the value that would maximize F1 score on the 80,000 training pixels, and then also evaluated that threshold on the 11,000 test pixels. The resulting threshold was set at NDVI $\approx 0.03$ and produced a Test F1 score of 0.93.

Meanwhile, the FNN learned the following classification threshold.

As can be seen, across most of the input space the machine learned threshold stays fairly close to the NDVI threshold. However, it is not a linear function of either Red or NIR, and it diverges most significantly from the linear threshold in the bottom-left corner, where both Red and NIR values are close to zero. Pixels in this range it classifies as background, while the NDVI threshold would classify them as vegetation. Interestingly, on the test set, the FNN threshold also produces an F1 score of 0.93, so on this particular dataset it doesn't improve that evaluation metric! (We look more closely at this and other metrics in the Jupyter notebook.) To understand why it makes this deviation then, we need to look at some actual images.

The first image we'll look at is taken from the Train dataset. All pixels in that bottom-left corner of the NDVI heatmap, where the NDVI threshold would classify them as plants while the FNN threshold would classify them as background, are colored in either orange or magenta -- orange if the FNN threshold is correct and they are actually background, magenta if the NDVI threshold is correct and they are actually plant life.

In this image, the FNN model is obviously correct for almost all pixels where the two thresholds diverge. This is an extreme case, and we will see images later where the two classes are more balanced, but in general the FNN model will have more correct predictions simply because there is more background than plant life (remember, we trained and evaluated both thresholds on a balanced dataset of plant and background pixels, but in actual photos we will generally have far more background pixels than plants, at least in the given dataset).

More interesting, almost all the highlighted pixels are in the shaded regions of the image. This gives us a very good indication of what that bottom-left corner of the NDVI heatmap corresponds to--shadows, where both Red and NIR reflectance is low. In these shadowed regions, the NDVI threshold obviously over-predicts plant life, as indicated by the divergences between its predictions and the (in this case correct) FNN predictions. We'll see a similar example in the next image.

This image is also taken from the Train dataset, and in this case actually all divergent predictions are background pixels (misclassified as plant life by the NDVI threshold). I include it along with the above image because it isn't clear to me whether the misclassified pixels are darker because of shadow, or if they represent darker soil dug up by the apparent disturbance of the ground along the bottom of the image (or both). It's one of the few images in either dataset where differences in threshold predictions aren't obviously limited to shaded regions.$^{b}$

($^{b}$I should clarify here that there are other divergent predictions between the two thresholds that aren't colored in--we are only highlighting pixels in the bottom-left corner of the NDVI heatmap, since this is where the two thresholds most clearly diverge.)

Now we come to an image where the two thresholds again diverge in their predictions on the shaded regions, but in this case some of those divergent pixels are in fact plant life (misclassified by the FNN threshold as background)! The right hand image shows the ground truth segmentation (background in black, vegetation in white), where it is easier to see the magenta pixels corresponding to plant life, correctly identified by the NDVI threshold but misidentified by the FNN threshold. This image is taken from the Test dataset, which by random chance seems to have more images than the Train dataset where shadowed plant life falls within that bottom-left corner of the NDVI heatmap, which might explain why the FNN does not perform as well on the Test dataset (equal to the NDVI threshold in terms of F1).

This is another image from the Test dataset, where again some of the shaded pixels where the two thresholds disagree are plant life and some are background. In this case, I would draw attention to some of the orange pixels near the bottom, which are annotated (by humans) as background, but based on the shapes they form look like they might be leaves of grass (visible in other parts of the image). It's possible that these were actually mis-annotated and should be plant life, as would be predicted by the NDVI threshold (though not the FNN threshold, trained on similar human annotations).

The final two images (shown together) are both of brightly, evenly-lit fields, and both have almost no divergent predictions from that bottom-left corner of the heatmap, reinforcing our interpretation of that divergence as the effect of shadow. The second image does however have some shadows cast by the plants themselves--why these don't fall into that divergent region isn't immediately clear, and suggests that there are additional factors affecting Red and NIR reflectance even in shadow.

## Discussion

Though this is a preliminary study

# Pilot Study: Machine Learning NDVI

# Setup

In [ ]:
import os
import cv2
import gdown
import torch
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.path import Path
from matplotlib.colors import to_rgba
from matplotlib.patches import PathPatch
import plotly.graph_objects as go
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_curve, precision_recall_fscore_support

In [ ]:
seed = 0
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def get_activations(model, name):
    '''
    params:
        name (string): Name of layer to store activations

    returns:
        None, writes activations to model dictionary "activations"
    '''
    def hook(module, input, output):
        model.activations[name] = output.detach().clone()
    return hook

def get_partitions(model, x_span, y_span):
    '''
    params:
        x_span (tuple): Tuple of the form (x_min, x_max, x_samples) indicating the minimum and maximum x-values
                        over which to calculate partitions and the number of samples within that range.
        y_span (tuple): Tuple of the form (y_min, y_max, y_samples), same purpose as for x_span but for y-dimension.

    returns:
        partitions (dict): Dictionary of {layer:vertices} storing vertices that can be used to graph the partitions
                           drawn by each layer's neurons.
    '''
    model.activations = dict()
    partitions = dict()
    x_span = np.linspace(*x_span)
    y_span = np.linspace(*y_span)
    meshgrid = np.meshgrid(x_span, y_span)
    uniform_input = torch.tensor(np.stack(
        [meshgrid[0].reshape(-1), meshgrid[1].reshape(-1)], 1), dtype=torch.float32).to(device)
    hooks = []
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear) or isinstance(module, torch.nn.Conv2d):
            hooks.append(module.register_forward_hook(get_activations(model, name)))
    with torch.no_grad():
        model.forward(uniform_input)
    for hook in hooks:
        hook.remove()
    for layer, activation in model.activations.items():
        paths = [plt.contour(
            meshgrid[0], meshgrid[1], activation[:,i].reshape(meshgrid[0].shape).cpu(), [0]
        ) for i in range(activation.shape[1])]
        paths = [path.get_paths()[0] for path in paths]
        plt.close()
        paths = [path.vertices[:-1] for path in paths]
        partitions[layer] = paths

    return partitions

# Task 1: Learning NDVI via Regression

In [ ]:
class NDVIDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx,:2]
        y = self.data[idx, 2:]
        return x, y

In [ ]:
# Create 256 x 256 uniform meshgrid of Red and NIR values between 0 and 1
red_values = np.linspace(0, 1, 256, dtype=np.float32)
nir_values = np.linspace(0, 1, 256, dtype=np.float32)
Red, NIR = np.meshgrid(red_values, nir_values)

# Calculate NDVI for each combination of Red and NIR values
sum_bands = NIR + Red
sum_bands[sum_bands == 0] = np.finfo(float).eps # Set a small epsilon to avoid division by zero
NDVI = (NIR - Red) / sum_bands

# Stack Red, NIR, and NDVI values and flatten into 40401 x 3 shape
data = np.dstack([Red, NIR, NDVI]).reshape((-1,3))
ndvi_ds = NDVIDataset(data)
ndvi_dl = DataLoader(ndvi_ds, batch_size=256, shuffle=True)

In [ ]:
# Graph NDVI as a surface above Red and NIR values
surface_trace = go.Surface(
    x=Red,
    y=NIR,
    z=NDVI,
    colorscale='Viridis',
    cmin=-1, cmax=1,
    showscale=False,
    name='NDVI Surface',
    hovertemplate='Red:   %{x:.2f}<br>NIR:   %{y:.2f}<br>NDVI: %{z:.2f}<extra></extra>')

layout = go.Layout(
    title=dict(
        text='NDVI 3D Plot',
        x=0.5,
        xanchor='center'),
    scene=dict(
        xaxis_title='Red',
        yaxis_title='NIR',
        zaxis_title='NDVI',
        xaxis_range=[0, 1],
        yaxis_range=[0, 1],
        zaxis_range=[-1, 1],
        aspectmode='cube',
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=0.8))
        )
    )

fig = go.Figure(data=surface_trace, layout=layout)
fig.show()

In [ ]:
if isinstance(seed, int):
    torch.manual_seed(seed)
hidden_layers = 3
width = 30

relu = torch.nn.ReLU()
tanh = torch.nn.Tanh()
input = torch.nn.Linear(in_features=2, out_features=width)
regressor = torch.nn.Linear(in_features=width, out_features=1)
layers = [input, relu]
for i in range(hidden_layers):
  layers.append(torch.nn.Linear(in_features=width, out_features=width))
  layers.append(relu)
layers.append(regressor)
layers.append(tanh)
regression_model = torch.nn.Sequential(*layers).to(device)

loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.AdamW(regression_model.parameters(), lr=1e-4)

print('Training Loss')
for epoch in range(100):
    epoch_loss = 0.0
    for input, truth in ndvi_dl:
        input, truth = input.to(device), truth.to(device)
        optimizer.zero_grad()
        pred = regression_model(input)
        loss = loss_fn(pred, truth)
        epoch_loss += loss.item()
        loss.backward()
        optimizer.step()
    if (epoch+1) % 10 == 0:
        print(f'{epoch+1:<3} {loss / len(ndvi_dl)}')

In [ ]:
inputs = torch.from_numpy(data[:,:2]).to(torch.float32).to(device)
NDVI_pred = regression_model(inputs).reshape((256,256)).detach().cpu().numpy()

# Graph NDVI as a surface above Red and NIR values
surface_trace = go.Surface(
    x=Red,
    y=NIR,
    z=NDVI_pred,
    colorscale='Viridis',
    cmin=-1, cmax=1,
    showscale=False,
    name='NDVI Surface',
    hovertemplate='Red:   %{x:.2f}<br>NIR:   %{y:.2f}<br>NDVI: %{z:.2f}<extra></extra>')

layout = go.Layout(
    title=dict(
        text='NDVI 3D Prediction',
        x=0.5,
        xanchor='center'),
    scene=dict(
        xaxis_title='Red',
        yaxis_title='NIR',
        zaxis_title='NDVI',
        xaxis_range=[0, 1],
        yaxis_range=[0, 1],
        zaxis_range=[-1, 1],
        aspectmode='cube',
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=0.8))
        )
    )

fig = go.Figure(data=surface_trace, layout=layout)

# Uncomment to highlight nonlinearities in the predicted function
'''
partitions = get_partitions(regression_model, x_span=(0,1,256), y_span=(0,1,256))

for partition in partitions.values():
    for k in range(len(partition)):
        inputs = torch.from_numpy(partition[k]).to(torch.float32)
        with torch.no_grad():
            outputs = regression_model(inputs).detach().flatten().numpy()
        fig.add_trace(go.Scatter3d(
            x=partition[k][:,0],
            y=partition[k][:,1],
            z=outputs,
            showlegend=False,
            hoverinfo='none',
            mode='lines',
            line=dict(
                color='black',
                width=5,
                dash='dash')
            ))
'''
fig.show()

# Task 2: Learning NDVI Thresholds via Classification

In [ ]:
# Create 256 x 256 uniform meshgrid of Red and NIR values between 0 and 1
red_values = np.linspace(0, 1, 256, dtype=np.float32)
nir_values = np.linspace(0, 1, 256, dtype=np.float32)
Red, NIR = np.meshgrid(red_values, nir_values)

# Calculate NDVI for each combination of Red and NIR values
sum_bands = NIR + Red
sum_bands[sum_bands == 0] = np.finfo(float).eps # Set a small epsilon to avoid division by zero
NDVI = (NIR - Red) / sum_bands

# Graph threshold against NDVI heatmap and test set samples
fig, axs = plt.subplots(nrows=1, ncols=1, figsize=(6,5))
heatmap = axs.imshow(NDVI, extent=[0,1,0,1], origin='lower')
contour = axs.contour(NDVI, levels=[-0.5,0,0.5], extent=[0,1,0,1], colors='black', linestyles='--')
clabels = axs.clabel(contour, fmt='%.1f', inline=True, inline_spacing=6, manual=[(0.9,0.3),(0.8,0.8),(0.3,0.9)])
axs.set_xlabel('Red')
axs.set_ylabel('NIR')
axs.set_title('NDVI Thresholds')
cbar = plt.colorbar(heatmap, ax=axs, ticks=[-1,0,1], label='NDVI')
plt.tight_layout()
plt.show()

In [ ]:
url = 'https://drive.google.com/uc?id=1NPp_AGew_wjJW0T5CZRj-OLmG9Vnv9x2'
target = '/content/DeepIndices_Dataset.zip'
if not os.path.exists(target):
    gdown.download(url, target)

data_folder = '/content/airphen-vegetation'
if not os.path.isdir(data_folder):
    with zipfile.ZipFile(target, 'r') as zip_ref:
        zip_ref.extractall('/content')

In [ ]:
class PixelDataset(Dataset):
    def __init__(self, data_folder, indices, num_samples=1000, seed=None):
        rng = np.random.default_rng(seed)
        self.pixels = []
        i = 0
        for folder1 in sorted(os.listdir(data_folder)):
            for folder2 in sorted(os.listdir(os.path.join(data_folder, folder1))):
                if i in indices:
                    Red = cv2.imread(os.path.join(data_folder, folder1, folder2, '2.png'), cv2.IMREAD_GRAYSCALE).flatten()
                    NIR = cv2.imread(os.path.join(data_folder, folder1, folder2, '5.png'), cv2.IMREAD_GRAYSCALE).flatten()
                    GT = cv2.imread(os.path.join(data_folder, folder1, folder2, 'gt.png'), cv2.IMREAD_GRAYSCALE).flatten()
                    GT[GT>=128], GT[GT<128] = 255, 0
                    image = np.stack([Red,NIR,GT], axis=-1)
                    if num_samples is None:
                        samples = min((GT==255).sum(), (GT==0).sum())
                        vegetation = rng.choice(image[GT==255], size=samples, replace=False)
                        background = rng.choice(image[GT==0], size=samples, replace=False)
                    else:
                        samples = min(num_samples//2, (GT==255).sum(), (GT==0).sum())
                        vegetation = rng.choice(image[GT==255], size=samples, replace=False)
                        background = rng.choice(image[GT==0], size=samples, replace=False)
                    self.pixels.append(np.concat([vegetation, background]))
                i += 1

        self.pixels = np.concat(self.pixels)
        self.pixels = np.divide(self.pixels, 255, dtype=np.float32)

    def __len__(self):
        return len(self.pixels)

    def __getitem__(self, idx):
        x = self.pixels[idx,:2]
        y = self.pixels[idx, 2:]
        return x, y

In [ ]:
# Set number of samples to draw from each image
num_samples = 1000

# Randomly select 80 images for Train dataset and 11 images for Test dataset
rng = np.random.default_rng(seed)
test_idx = set(rng.choice(91, size=11, replace=False).tolist())
train_idx = set(range(91)) - test_idx
print('Test Index:', sorted(test_idx))

# Create Train and Test datasets and dataloaders
pixel_train_ds = PixelDataset(data_folder, train_idx, num_samples, seed=seed)
pixel_test_ds = PixelDataset(data_folder, test_idx, num_samples, seed=seed)
pixel_train_dl = DataLoader(pixel_train_ds, batch_size=200, shuffle=True)
pixel_test_dl = DataLoader(pixel_test_ds, batch_size=100, shuffle=True)
train_pixels = pixel_train_ds.pixels
test_pixels = pixel_test_ds.pixels
print('Train pixels:', len(pixel_train_ds), 'Test pixels:', len(pixel_test_ds))

In [ ]:
# Calculate NDVI over all pixels in Train dataset
red_train = train_pixels[:,0]
nir_train = train_pixels[:,1]
labels_train = train_pixels[:,2]
sum_train = nir_train + red_train
sum_train[sum_train == 0] = np.finfo(float).eps
ndvi_train = (nir_train - red_train) / sum_train

# Calculate NDVI threshold that maximizes F1 on Train dataset
precision, recall, thresholds = precision_recall_curve(labels_train, ndvi_train)
f1_train = 2 * (precision * recall) / (precision + recall)
idx = np.argmax(f1_train)
threshold = thresholds[idx]
precision_train = precision[idx]
recall_train = recall[idx]
f1_train = f1_train[idx]

# Calculate NDVI over all pixels in Test dataset
red_test = test_pixels[:,0]
nir_test = test_pixels[:,1]
labels_test = test_pixels[:,2]
sum_test = nir_test + red_test
sum_test[sum_test == 0] = np.finfo(float).eps
ndvi_test = (nir_test - red_test) / sum_test

# Calculate F1 score on Test dataset using NDVI threshold
preds_test = ndvi_test >= threshold
precision_test, recall_test, f1_test, _ = precision_recall_fscore_support(labels_test, preds_test, average='binary')

# Print threshold, Train and Test metrics
print(f'Threshold: {threshold:.2f}')
print()
print('---Train Set---')
print(f'Precision: {precision_train:.2f}')
print(f'Recall:    {recall_train:.2f}')
print(f'F1 Score:  {f1_train:.2f}')
print()
print('----Test Set---')
print(f'Precision: {precision_test:.2f}')
print(f'Recall:    {recall_test:.2f}')
print(f'F1 Score:  {f1_test:.2f}')

In [ ]:
# Create 256 x 256 uniform meshgrid of Red and NIR values between 0 and 1
red_values = np.linspace(0, 1, 256, dtype=np.float32)
nir_values = np.linspace(0, 1, 256, dtype=np.float32)
Red, NIR = np.meshgrid(red_values, nir_values)

# Calculate NDVI for each combination of Red and NIR values
sum_bands = NIR + Red
sum_bands[sum_bands == 0] = np.finfo(float).eps # Set a small epsilon to avoid division by zero
NDVI = (NIR - Red) / sum_bands

# Graph threshold against NDVI heatmap and test set samples
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(10,4.5), gridspec_kw={'width_ratios': [1, .8]})
heatmap = axs[0].imshow(NDVI, extent=[0,1,0,1], origin='lower')
contour = axs[0].contour(NDVI, levels=[threshold], extent=[0,1,0,1], colors='black', linestyles='--')
ndvi_boundary = contour.get_paths()[0]
ndvi_boundary = ndvi_boundary.vertices[:-1]
axs[0].clabel(contour, fmt='%.2f', inline=True, inline_spacing=6, manual=[(0.8,0.8)])
tp_tn = test_pixels[labels_test==preds_test]
colors = ['lawngreen' if label==1 else 'darkblue' for label in tp_tn[:,2]]
axs[1].scatter(tp_tn[:,0], tp_tn[:,1], s=2, c=colors)
fp_fn = test_pixels[labels_test!=preds_test]
colors = ['lawngreen' if label==1 else 'darkblue' for label in fp_fn[:,2]]
axs[1].scatter(fp_fn[:,0], fp_fn[:,1], s=2, c=colors)
axs[1].plot(ndvi_boundary[:,0], ndvi_boundary[:,1], color='white', linewidth=3)
axs[1].plot(ndvi_boundary[:,0], ndvi_boundary[:,1], color='black', linestyle='--')
axs[1].set_aspect('equal')
axs[1].set_xlim((0,1))
axs[1].set_ylim((0,1))
for ax in axs:
    ax.set_xlabel('Red')
    ax.set_ylabel('NIR')
    ax.set_xticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
axs[0].set_title('NDVI Heatmap')
axs[1].set_title('Test Set')
cbar = plt.colorbar(heatmap, ax=axs[0], ticks=[-1,0,1])
cbar.set_label('NDVI', labelpad=-47)
plt.tight_layout()
plt.show()

In [ ]:
if isinstance(seed, int):
    torch.manual_seed(seed)
hidden_layers = 3
width = 30

relu = torch.nn.ReLU()
input = torch.nn.Linear(in_features=2, out_features=width)
classifier = torch.nn.Linear(in_features=width, out_features=1)
layers = [input, relu]
for i in range(hidden_layers):
  layers.append(torch.nn.Linear(in_features=width, out_features=width))
  layers.append(relu)
layers.append(classifier)
classification_model = torch.nn.Sequential(*layers).to(device)

loss_fn = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(classification_model.parameters(), lr=1e-4)

print('Training Loss')
for epoch in range(10):
    epoch_loss = 0.0
    for input, truth in pixel_train_dl:
        input, truth = input.to(device), truth.to(device)
        optimizer.zero_grad()
        pred = classification_model(input)
        loss = loss_fn(pred, truth)
        epoch_loss += loss.item()
        loss.backward()
        optimizer.step()
    print(f'{epoch+1:<3} {loss / len(pixel_train_dl)}')
print()

# Run model predictions on Train dataset
inputs_train = torch.from_numpy(train_pixels[:,:2]).to(device)
labels_train = train_pixels[:,2]
with torch.no_grad():
    preds_train = classification_model(inputs_train) >= 0
preds_train = preds_train[:,0].detach().cpu().numpy()
precision_train, recall_train, f1_train, _ = precision_recall_fscore_support(labels_train, preds_train, average='binary')

# Run model predictions on Test dataset
inputs_test = torch.from_numpy(test_pixels[:,:2]).to(device)
labels_test = test_pixels[:,2]
with torch.no_grad():
    preds_test = classification_model(inputs_test) >= 0
preds_test = preds_test[:,0].detach().cpu().numpy()
precision_test, recall_test, f1_test, _ = precision_recall_fscore_support(labels_test, preds_test, average='binary')

# Print Train and Test metrics
print('---Train Set---')
print(f'Precision: {precision_train:.2f}')
print(f'Recall:    {recall_train:.2f}')
print(f'F1 Score:  {f1_train:.2f}')
print()
print('----Test Set---')
print(f'Precision: {precision_test:.2f}')
print(f'Recall:    {recall_test:.2f}')
print(f'F1 Score:  {f1_test:.2f}')

In [ ]:
# Create 256 x 256 uniform meshgrid of Red and NIR values between 0 and 1
red_values = np.linspace(0, 1, 256, dtype=np.float32)
nir_values = np.linspace(0, 1, 256, dtype=np.float32)
Red, NIR = np.meshgrid(red_values, nir_values)

# Calculate NDVI for each combination of Red and NIR values
sum_bands = NIR + Red
sum_bands[sum_bands == 0] = np.finfo(float).eps # Set a small epsilon to avoid division by zero
NDVI = (NIR - Red) / sum_bands

# Calculate model partitions on input space
partitions = get_partitions(classification_model, x_span=(0,1,256), y_span=(0,1,256))
model_boundary = list(partitions.values())[-1][0]

# Graph threshold against NDVI heatmap and test set samples
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(10,4.5), gridspec_kw={'width_ratios': [1, .8]})
heatmap = axs[0].imshow(NDVI, extent=[0,1,0,1], origin='lower')
axs[0].plot(model_boundary[:,0], model_boundary[:,1], color='black', linestyle='--')
tp_tn = test_pixels[labels_test==preds_test]
colors = ['lawngreen' if label==1 else 'darkblue' for label in tp_tn[:,2]]
axs[1].scatter(tp_tn[:,0], tp_tn[:,1], s=2, c=colors)
fp_fn = test_pixels[labels_test!=preds_test]
colors = ['lawngreen' if label==1 else 'darkblue' for label in fp_fn[:,2]]
axs[1].scatter(fp_fn[:,0], fp_fn[:,1], s=2, c=colors)
axs[1].plot(model_boundary[:,0], model_boundary[:,1], color='white', linewidth=3)
axs[1].plot(model_boundary[:,0], model_boundary[:,1], color='black', linestyle='--')
axs[1].set_aspect('equal')
axs[1].set_xlim((0,1))
axs[1].set_ylim((0,1))
for ax in axs:
    ax.set_xlabel('Red')
    ax.set_ylabel('NIR')
    ax.set_xticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
axs[0].set_title('NDVI Heatmap')
axs[1].set_title('Test Set')
cbar = plt.colorbar(heatmap, ax=axs[0], ticks=[-1,0,1])
cbar.set_label('NDVI', labelpad=-47)
plt.tight_layout()
plt.show()

In [ ]:
# Create 256 x 256 uniform meshgrid of Red and NIR values between 0 and 1
red_values = np.linspace(0, 1, 256, dtype=np.float32)
nir_values = np.linspace(0, 1, 256, dtype=np.float32)
Red, NIR = np.meshgrid(red_values, nir_values)

# Calculate NDVI for each combination of Red and NIR values
sum_bands = NIR + Red
sum_bands[sum_bands == 0] = np.finfo(float).eps # Set a small epsilon to avoid division by zero
NDVI = (NIR - Red) / sum_bands

# Graph machine learned boundary and ndvi threshold against each other
fig, axs = plt.subplots(nrows=1, ncols=1, figsize=(6,5))
heatmap = axs.imshow(NDVI, extent=[0,1,0,1], origin='lower')
axs.plot(ndvi_boundary[:,0], ndvi_boundary[:,1], color='white', linestyle='--')
axs.plot(model_boundary[:,0], model_boundary[:,1], color='black', linestyle='--')
axs.set_xlabel('Red')
axs.set_ylabel('NIR')
axs.set_title('NDVI Thresholds')
cbar = plt.colorbar(heatmap, ax=axs, ticks=[-1,0,1], label='NDVI')
plt.tight_layout()
plt.show()

In [ ]:
# Interpolate model and ndvi thresholds so they can be compared on every sampled value of Red and NIR
red_samples = np.unique(np.sort(np.concatenate((ndvi_boundary[:,0], model_boundary[:,0]))))
nir_samples_ndvi = np.interp(red_samples, ndvi_boundary[:,0], ndvi_boundary[:,1])
nir_samples_model = np.interp(red_samples, model_boundary[:,0], model_boundary[:,1])
ndvi_boundary_interp = np.stack([red_samples, nir_samples_ndvi], axis=-1)
model_boundary_interp = np.stack([red_samples, nir_samples_model], axis=-1)

# Find index where model and ndvi thresholds cross
cross_idx = np.nonzero(np.diff(np.sign(nir_samples_model - nir_samples_ndvi)))[0][0]

# Draw path around bottom left corner where model and ndvi thresholds diverge
diff = Path(np.concat([model_boundary_interp[:cross_idx+1], ndvi_boundary_interp[cross_idx::-1]]))
#diff_patch1 = PathPatch(diff, edgecolor='orangered', linewidth=3, linestyle='--', fill=False, zorder=3)
diff_patch2 = PathPatch(diff, edgecolor='orangered', linewidth=3, linestyle='--', fill=False, zorder=3)

# Calculate how many pixels within the difference are vegetation vs background
idx = diff.contains_points(train_pixels[:,:2])
diff_train = labels_train[idx]
pos_train = diff_train.sum() / len(diff_train)
neg_train = 1 - pos_train
idx = diff.contains_points(test_pixels[:,:2])
diff_test = labels_test[idx]
pos_test = diff_test.sum() / len(diff_test)
neg_test = 1 - pos_test

# Create 256 x 256 uniform meshgrid of Red and NIR values between 0 and 1
red_values = np.linspace(0, 1, 256, dtype=np.float32)
nir_values = np.linspace(0, 1, 256, dtype=np.float32)
Red, NIR = np.meshgrid(red_values, nir_values)

# Calculate NDVI for each combination of Red and NIR values
sum_bands = NIR + Red
sum_bands[sum_bands == 0] = np.finfo(float).eps # Set a small epsilon to avoid division by zero
NDVI = (NIR - Red) / sum_bands

# Calculate model partitions on input space
partitions = get_partitions(classification_model, x_span=(0,1,256), y_span=(0,1,256))
model_boundary = list(partitions.values())[-1][0]

# Graph threshold against NDVI heatmap and test set samples
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(10,4.5), gridspec_kw={'width_ratios': [1, .8]})
heatmap = axs[0].imshow(NDVI, extent=[0,1,0,1], origin='lower')
axs[0].plot(ndvi_boundary[:,0], ndvi_boundary[:,1], color='white', linestyle='--')
axs[0].plot(model_boundary[:,0], model_boundary[:,1], color='black', linestyle='--')
#axs[0].add_patch(diff_patch1)
tp_tn = test_pixels[labels_test==preds_test]
colors = ['lawngreen' if label==1 else 'darkblue' for label in tp_tn[:,2]]
axs[1].scatter(tp_tn[:,0], tp_tn[:,1], s=2, c=colors)
fp_fn = test_pixels[labels_test!=preds_test]
colors = ['lawngreen' if label==1 else 'darkblue' for label in fp_fn[:,2]]
axs[1].scatter(fp_fn[:,0], fp_fn[:,1], s=2, c=colors)
axs[1].plot(ndvi_boundary[:,0], ndvi_boundary[:,1], color='white', linestyle='--')
axs[1].plot(model_boundary[:,0], model_boundary[:,1], color='black', linestyle='--')
axs[1].add_patch(diff_patch2)
axs[1].set_aspect('equal')
axs[1].set_xlim((0,1))
axs[1].set_ylim((0,1))
for ax in axs:
    ax.set_xlabel('Red')
    ax.set_ylabel('NIR')
    ax.set_xticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
axs[0].set_title('NDVI Heatmap')
axs[1].set_title('Test Set')
cbar = plt.colorbar(heatmap, ax=axs[0], ticks=[-1,0,1])
cbar.set_label('NDVI', labelpad=-47)
plt.tight_layout()
plt.show()
print(f'Percent of train set pixels within divergent region: {len(diff_train)/len(labels_train):.1%}')
print(f'Within the divergent region, percent vegetation: {pos_train:.1%}  vs  background: {neg_train:.1%}')
print()
print(f'Percent of test set pixels within divergent region: {len(diff_test)/len(labels_test):.1%}')
print(f'Within the divergent region, percent vegetation: {pos_test:.1%}  vs  background: {neg_test:.1%}')

## Qualitative Analysis

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, data_folder, indices):
        i = 0
        self.filenames = []
        for folder1 in sorted(os.listdir(data_folder)):
            for folder2 in sorted(os.listdir(os.path.join(data_folder, folder1))):
                if i in indices:
                    self.filenames.append(os.path.join(data_folder, folder1, folder2))
                i += 1

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        filename = self.filenames[idx]
        Blue = cv2.imread(os.path.join(filename, '0.png'), cv2.IMREAD_GRAYSCALE)
        Green = cv2.imread(os.path.join(filename, '1.png'), cv2.IMREAD_GRAYSCALE)
        Red = cv2.imread(os.path.join(filename, '2.png'), cv2.IMREAD_GRAYSCALE)
        NIR = cv2.imread(os.path.join(filename, '5.png'), cv2.IMREAD_GRAYSCALE)
        GT = cv2.imread(os.path.join(filename, 'gt.png'), cv2.IMREAD_GRAYSCALE)
        GT = (GT>=128)[:,:,np.newaxis]

        RGB = np.stack([Red, Green, Blue], axis=-1)
        R_NIR = np.stack([Red, NIR], axis=-1)

        RGB = np.divide(RGB, 255, dtype=np.float32)
        R_NIR = np.divide(R_NIR, 255, dtype=np.float32)

        return RGB, R_NIR, GT

In [ ]:
image_train_ds = ImageDataset(data_folder, train_idx)
image_test_ds = ImageDataset(data_folder, test_idx)
print('Test Index:', sorted(test_idx))

In [ ]:
idx = 75
filename = image_train_ds.filenames[idx]
RGB, R_NIR, GT = image_train_ds[idx]
H, W, C = R_NIR.shape
pixels = R_NIR.reshape((H*W,C))
mask = diff.contains_points(pixels).reshape((H,W,1))
veg_color = np.array(to_rgba('magenta'))
back_color = np.array(to_rgba('orangered'))
veg_overlay = np.ones((H,W,4)) * veg_color * GT * mask
back_overlay = np.ones((H,W,4)) * back_color * ~GT * mask
percent_mask = mask.sum() / (H*W)
percent_veg = (GT * mask).sum() / mask.sum()
percent_back = (~GT * mask).sum() / mask.sum()

fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(12,4.5))
axs[0].imshow(RGB)
axs[1].imshow(RGB)
axs[1].imshow(veg_overlay)
axs[1].imshow(back_overlay)
for ax in axs:
    ax.axis('off')
plt.tight_layout()
plt.show()
print(filename)
print(f'Percent pixels in divergent region: {percent_mask:.1%}')
print(f'Percent vegetation: {percent_veg:.1%}  Percent background: {percent_back:.1%}')

In [ ]:
idx = 46
filename = image_train_ds.filenames[idx]
RGB, R_NIR, GT = image_train_ds[idx]
H, W, C = R_NIR.shape
pixels = R_NIR.reshape((H*W,C))
mask = diff.contains_points(pixels).reshape((H,W,1))
veg_color = np.array(to_rgba('magenta'))
back_color = np.array(to_rgba('orangered'))
veg_overlay = np.ones((H,W,4)) * veg_color * GT * mask
back_overlay = np.ones((H,W,4)) * back_color * ~GT * mask
percent_mask = mask.sum() / (H*W)
percent_veg = (GT * mask).sum() / mask.sum()
percent_back = (~GT * mask).sum() / mask.sum()

fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(12,4.5))
axs[0].imshow(RGB)
axs[1].imshow(RGB)
axs[1].imshow(veg_overlay)
axs[1].imshow(back_overlay)
for ax in axs:
    ax.axis('off')
plt.tight_layout()
plt.show()
print(filename)
print(f'Percent pixels in divergent region: {percent_mask:.1%}')
print(f'Percent vegetation: {percent_veg:.1%}  Percent background: {percent_back:.1%}')

In [ ]:
idx = 0
filename = image_test_ds.filenames[idx]
RGB, R_NIR, GT = image_test_ds[idx]
H, W, C = R_NIR.shape
pixels = R_NIR.reshape((H*W,C))
mask = diff.contains_points(pixels).reshape((H,W,1))
veg_color = np.array(to_rgba('magenta'))
back_color = np.array(to_rgba('orangered'))
veg_overlay = np.ones((H,W,4)) * veg_color * GT * mask
back_overlay = np.ones((H,W,4)) * back_color * ~GT * mask
percent_mask = mask.sum() / (H*W)
percent_veg = (GT * mask).sum() / mask.sum()
percent_back = (~GT * mask).sum() / mask.sum()

fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(12,4.5))
axs[0].imshow(RGB)
axs[1].imshow(GT, cmap='grey')
for ax in axs:
    ax.imshow(veg_overlay)
    ax.imshow(back_overlay)
    ax.axis('off')
plt.tight_layout()
plt.show()
print(filename)
print(f'Percent pixels in divergent region: {percent_mask:.1%}')
print(f'Percent vegetation: {percent_veg:.1%}  Percent background: {percent_back:.1%}')

In [ ]:
idx = 1
filename = image_test_ds.filenames[idx]
RGB, R_NIR, GT = image_test_ds[idx]
H, W, C = R_NIR.shape
pixels = R_NIR.reshape((H*W,C))
mask = diff.contains_points(pixels).reshape((H,W,1))
veg_color = np.array(to_rgba('magenta'))
back_color = np.array(to_rgba('orangered'))
veg_overlay = np.ones((H,W,4)) * veg_color * GT * mask
back_overlay = np.ones((H,W,4)) * back_color * ~GT * mask
percent_mask = mask.sum() / (H*W)
percent_veg = (GT * mask).sum() / mask.sum()
percent_back = (~GT * mask).sum() / mask.sum()

fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(12,4.5))
axs[0].imshow(RGB)
axs[1].imshow(GT, cmap='grey')
for ax in axs:
    ax.imshow(veg_overlay)
    ax.imshow(back_overlay)
    ax.axis('off')
plt.tight_layout()
plt.show()
print(filename)
print(f'Percent pixels in divergent region: {percent_mask:.1%}')
print(f'Percent vegetation: {percent_veg:.1%}  Percent background: {percent_back:.1%}')

In [ ]:
idx = 4
filename = image_test_ds.filenames[idx]
RGB, R_NIR, GT = image_test_ds[idx]
H, W, C = R_NIR.shape
pixels = R_NIR.reshape((H*W,C))
mask = diff.contains_points(pixels).reshape((H,W,1))
veg_color = np.array(to_rgba('magenta'))
back_color = np.array(to_rgba('orangered'))
veg_overlay = np.ones((H,W,4)) * veg_color * GT * mask
back_overlay = np.ones((H,W,4)) * back_color * ~GT * mask
percent_mask = mask.sum() / (H*W)
percent_veg = (GT * mask).sum() / mask.sum()
percent_back = (~GT * mask).sum() / mask.sum()

fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(12,4.5))
axs[0].imshow(RGB)
axs[1].imshow(GT, cmap='grey')
for ax in axs:
    ax.imshow(veg_overlay)
    ax.imshow(back_overlay)
    ax.axis('off')
plt.tight_layout()
plt.show()
print(filename)
print(f'Percent pixels in divergent region: {percent_mask:.1%}')
print(f'Percent vegetation: {percent_veg:.1%}  Percent background: {percent_back:.1%}')

In [ ]:
idx = 14
filename = image_train_ds.filenames[idx]
RGB, R_NIR, GT = image_train_ds[idx]
H, W, C = R_NIR.shape
pixels = R_NIR.reshape((H*W,C))
mask = diff.contains_points(pixels).reshape((H,W,1))
veg_color = np.array(to_rgba('magenta'))
back_color = np.array(to_rgba('orangered'))
veg_overlay = np.ones((H,W,4)) * veg_color * GT * mask
back_overlay = np.ones((H,W,4)) * back_color * ~GT * mask
percent_mask = mask.sum() / (H*W)
percent_veg = (GT * mask).sum() / mask.sum()
percent_back = (~GT * mask).sum() / mask.sum()

fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(12,4.5))
axs[0].imshow(RGB)
axs[1].imshow(GT, cmap='grey')
for ax in axs:
    ax.imshow(veg_overlay)
    ax.imshow(back_overlay)
    ax.axis('off')
plt.tight_layout()
plt.show()
print(filename)
print(f'Percent pixels in divergent region: {percent_mask:.1%}')
print(f'Percent vegetation: {percent_veg:.1%}  Percent background: {percent_back:.1%}')

**Dataset Reference:**

Vayssade, Jehan-Antoine; Jones, Gawain; Paoli, Jean-Noël; Gée, Christelle, 2021, "Dataset used in DeepIndices", https://doi.org/10.15454/DSQC8N, Recherche Data Gouv, V2